In [ ]:
!pip install lightgbm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로드 완료")

✅ 라이브러리 로드 완료


In [ ]:
# ⚠️ DATA_DIR을 본인 폴더 경로로 수정하세요
DATA_DIR = '/content/drive/MyDrive/data'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"클래스 수: {train['target'].nunique()}")
print("\n--- Train 처음 5줄 ---")
train.head()

Train shape: (21693, 54)
Test shape:  (15004, 53)
클래스 수: 21

--- Train 처음 5줄 ---


,ID,X_01,X_02,X_03,X_04,X_05,X_06,X_07,X_08,X_09,...,X_44,X_45,X_46,X_47,X_48,X_49,X_50,X_51,X_52,target
0,TRAIN_00000,0.016,0.242994,0.538536,0.522295,0.374494,0.555348,0.639091,0.494800,0.584233,...,0.435885,0.555359,0.751714,0.376801,0.466993,0.527585,0.598101,0.312160,0.582797,0
1,TRAIN_00001,0.019,0.240380,0.517223,0.538976,0.371149,0.693825,0.663667,0.530931,0.577200,...,0.479859,0.693855,0.748955,0.356118,0.613461,0.508069,0.569814,0.313351,0.570513,20
2,TRAIN_00002,0.012,0.248946,0.547109,0.466713,0.415830,0.656887,0.681782,0.580773,0.527069,...,0.416115,0.656884,0.750059,0.417200,0.525393,0.551947,0.639860,0.342627,0.573139,1
3,TRAIN_00003,0.013,0.245877,0.527870,0.515534,0.379199,0.594391,0.663816,0.494931,0.581796,...,0.436761,0.594364,0.746297,0.374659,0.694290,0.532705,0.581142,0.316417,0.562656,19
4,TRAIN_00004,0.024,0.239237,0.566087,0.514384,0.378451,0.610543,0.644811,0.508567,0.593614,...,0.422407,0.610526,0.749565,0.372742,0.565549,0.535573,0.580484,0.314982,0.591692,15


In [ ]:
def feature_engineering(df):
    """FE 1차: EDA 기반 17개 피처 추가 (최종 채택)"""
    df = df.copy()
    sensors = [c for c in df.columns if c.startswith('X_')]

    # 1. 행 단위 기본 통계
    df['row_mean']   = df[sensors].mean(axis=1)
    df['row_std']    = df[sensors].std(axis=1)
    df['row_min']    = df[sensors].min(axis=1)
    df['row_max']    = df[sensors].max(axis=1)
    df['row_range']  = df['row_max'] - df['row_min']
    df['row_skew']   = df[sensors].skew(axis=1)
    df['row_median'] = df[sensors].median(axis=1)

    # 2. 핵심 센서 통계 (EDA에서 발견: X_19, X_37, X_40, X_11)
    core_sensors = ['X_19', 'X_37', 'X_40', 'X_11']
    df['core_mean'] = df[core_sensors].mean(axis=1)
    df['core_std']  = df[core_sensors].std(axis=1)
    df['core_max']  = df[core_sensors].max(axis=1)
    df['core_min']  = df[core_sensors].min(axis=1)

    # 3. 핵심 센서 페어 조합
    df['X19_X37_ratio'] = df['X_19'] / (df['X_37'] + 1e-6)
    df['X19_X40_ratio'] = df['X_19'] / (df['X_40'] + 1e-6)
    df['X19_X11_ratio'] = df['X_19'] / (df['X_11'] + 1e-6)
    df['X19_X37_diff']  = df['X_19'] - df['X_37']
    df['X19_X40_diff']  = df['X_19'] - df['X_40']
    df['X37_X40_diff']  = df['X_37'] - df['X_40']

    return df


# FE 적용
train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

print(f"원본 컬럼 수: {len(train.columns)}")
print(f"FE 후 컬럼 수: {len(train_fe.columns)}")
print(f"추가 피처: {len(train_fe.columns) - len(train.columns)}개")

원본 컬럼 수: 54
FE 후 컬럼 수: 71
추가 피처: 17개


In [ ]:
def evaluate_lgbm(X, y, name, n_splits=5):
    """LightGBM 5-fold OOF 평가"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    last_model = None

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=63,
            random_state=42,
            verbose=-1,
            n_jobs=-1
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

        preds = model.predict(X_val)
        scores.append(f1_score(y_val, preds, average='macro'))
        last_model = model

    mean_score = np.mean(scores)
    std_score = np.std(scores)
    print(f"  {name:40s}: {mean_score:.4f} (±{std_score:.4f})")
    return mean_score, last_model

print("✅ 평가 함수 정의 완료")

✅ 평가 함수 정의 완료


In [ ]:
# ============================================
# 무쓸모 센서 후보 (EDA 분리력 분석 기반)
# ============================================
useless_all  = ['X_01', 'X_34', 'X_48', 'X_03', 'X_23']  # 5개 모두
useless_no34 = ['X_01', 'X_48', 'X_03', 'X_23']           # X_34 빼고 (importance 높음)
useless_x01  = ['X_01']                                    # 진짜 노이즈만

# ============================================
# 실험 실행
# ============================================
print("=" * 70)
print("🧪 무쓸모 센서 제거 실험")
print("=" * 70)

y = train['target'].values

# 실험 D: 기준 (FE 1차 전체)
X_base = train_fe.drop(columns=['ID', 'target']).values
print(f"\n[기준] FE 1차 전체 ({X_base.shape[1]}개 피처)")ㅌ
score_D, _ = evaluate_lgbm(X_base, y, "D. 기준 (FE 1차 전체)")

# 실험 A: 5개 모두 제거
X_A = train_fe.drop(columns=['ID', 'target'] + useless_all).values
print(f"\n[A] 무쓸모 5개 모두 제거 ({X_A.shape[1]}개 피처)")
score_A, _ = evaluate_lgbm(X_A, y, "A. 5개 모두 제거")

# 실험 B: X_34만 유지
X_B = train_fe.drop(columns=['ID', 'target'] + useless_no34).values
print(f"\n[B] X_34 유지, 4개 제거 ({X_B.shape[1]}개 피처)")
score_B, _ = evaluate_lgbm(X_B, y, "B. X_34 유지, 4개 제거")

# 실험 C: X_01만 제거
X_C = train_fe.drop(columns=['ID', 'target'] + useless_x01).values
print(f"\n[C] X_01만 제거 ({X_C.shape[1]}개 피처)")
score_C, _ = evaluate_lgbm(X_C, y, "C. X_01만 제거")

# ============================================
# 결과 종합
# ============================================
print("\n" + "=" * 70)
print("📊 결과 종합")
print("=" * 70)
results = {
    "D. 기준 (FE 1차 전체)":     score_D,
    "A. 5개 모두 제거":          score_A,
    "B. X_34 유지, 4개 제거":    score_B,
    "C. X_01만 제거":            score_C,
}

for name, score in results.items():
    diff = score - score_D
    marker = "🥇" if score == max(results.values()) else "  "
    print(f"{marker} {name:30s}: {score:.4f}  ({diff:+.4f})")

# 결론
print("\n💡 분석:")
best_name = max(results, key=results.get)
best_score = results[best_name]
print(f"  🏆 베스트: {best_name} ({best_score:.4f})")

if best_name == "D. 기준 (FE 1차 전체)":
    print("  ⚠️ 어떤 제거도 효과 없음 → FE 1차 전체 유지 권장")
elif best_score - score_D > 0.001:
    print(f"  ✅ 유의미한 개선! ({best_score - score_D:+.4f})")
else:
    print(f"  ⚠️ 미세한 차이 → 노이즈일 수 있음, 단순성 위해 전체 유지 권장")

🧪 무쓸모 센서 제거 실험

[기준] FE 1차 전체 (69개 피처)
  D. 기준 (FE 1차 전체)                        : 0.8045 (±0.0018)

[A] 무쓸모 5개 모두 제거 (64개 피처)
  A. 5개 모두 제거                             : 0.7801 (±0.0042)

[B] X_34 유지, 4개 제거 (65개 피처)
  B. X_34 유지, 4개 제거                       : 0.7954 (±0.0032)

[C] X_01만 제거 (68개 피처)
  C. X_01만 제거                             : 0.8050 (±0.0037)

📊 결과 종합
   D. 기준 (FE 1차 전체)              : 0.8045  (+0.0000)
   A. 5개 모두 제거                   : 0.7801  (-0.0245)
   B. X_34 유지, 4개 제거             : 0.7954  (-0.0092)
🥇 C. X_01만 제거                   : 0.8050  (+0.0005)

💡 분석:
  🏆 베스트: C. X_01만 제거 (0.8050)
  ⚠️ 미세한 차이 → 노이즈일 수 있음, 단순성 위해 전체 유지 권장


In [ ]:
def feature_engineering(df, drop_useless=True):
    """FE 1차 + X_01 제거 (최종)"""
    df = df.copy()

    # ⭐ X_01 제거 (진짜 노이즈)
    if drop_useless and 'X_01' in df.columns:
        df = df.drop(columns=['X_01'])

    sensors = [c for c in df.columns if c.startswith('X_')]

    # 1. 행 단위 기본 통계
    df['row_mean']   = df[sensors].mean(axis=1)
    df['row_std']    = df[sensors].std(axis=1)
    df['row_min']    = df[sensors].min(axis=1)
    df['row_max']    = df[sensors].max(axis=1)
    df['row_range']  = df['row_max'] - df['row_min']
    df['row_skew']   = df[sensors].skew(axis=1)
    df['row_median'] = df[sensors].median(axis=1)

    # 2. 핵심 센서 통계
    core_sensors = ['X_19', 'X_37', 'X_40', 'X_11']
    df['core_mean'] = df[core_sensors].mean(axis=1)
    df['core_std']  = df[core_sensors].std(axis=1)
    df['core_max']  = df[core_sensors].max(axis=1)
    df['core_min']  = df[core_sensors].min(axis=1)

    # 3. 핵심 센서 페어 조합
    df['X19_X37_ratio'] = df['X_19'] / (df['X_37'] + 1e-6)
    df['X19_X40_ratio'] = df['X_19'] / (df['X_40'] + 1e-6)
    df['X19_X11_ratio'] = df['X_19'] / (df['X_11'] + 1e-6)
    df['X19_X37_diff']  = df['X_19'] - df['X_37']
    df['X19_X40_diff']  = df['X_19'] - df['X_40']
    df['X37_X40_diff']  = df['X_37'] - df['X_40']

    return df